## Knowledge Distillation

Knowledge Distillation(KD, 지식 증류)는 고성능의 모델(Teacher)에서 지식을 전달 받아 상대적으로 간단한 모델(Student)을 학습시키는 방법

Teacher 모델이 내부 구조, 파라미터를 공개 여부에 따라 White-box, Black-box, Gray-box로 구분됩니다. Black-box는 결과만 확인 가능한 경우며  <br>
White-box는 모델의 내부 구조나 파라미터를 전부 알 수 있는 경우입니다. Gray-box는 그 중간으로 일부만 공개되어 있는 경우입니다. <br>
이러한 Teacher 모델의 특징에 의해 KD에 활용할 수 있는 정보의 종류가 달라지게 됩니다.


#### Reference:
https://docs.pytorch.org/tutorials/beginner/knowledge_distillation_tutorial.html

https://github.com/NoCodeProgram/deepLearning/blob/main/transformer/KD_toy.ipynb

<br>

----
Pytorch Knowledge Distillation 튜토리얼을 기반으로 다음 내용을 학습습합니다.

(1) 모델 클래스 수정 방법

: 모델의 은닉 표현(hidden representation)을 추출하고, 이를 추가적인 계산에 활용할 수 있도록 모델 클래스를 수정하는 방법을 배웁니다.

(2) PyTorch 학습 루프 수정 방법

: 분류를 위한 Cross-Entropy 손실 외에, 추가적인 손실 함수를 포함하도록 PyTorch의 일반적인 학습 루프를 수정하는 방법을 배웁니다.

(3) 경량화 모델의 성능 향상 방법

보다 복잡하고 성능이 좋은 큰 모델(Teacher) 을 이용해, 작고 빠른 모델(Student) 의 성능을 향상시키는 방법을 배웁니다.

----

Packages import

In [28]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import torchvision.datasets as datasets
from torch.utils.data import DataLoader
from torch import Tensor

Torch Device

In [29]:
if torch.backends.mps.is_available():
  my_device = torch.device('mps')
elif torch.cuda.is_available():
  my_device = torch.device('cuda')
else:
  my_device = torch.device('cpu')

print(f"Using device: {my_device}")

Using device: cuda


#### CIFAR-10 dataset

- 10개의 클래스
- 32x32 픽셀 이미지
- 10,000개의 이미지
- 50,000개의 트레이닝 이미지
- 10,000개의 테스트 이미지

입력 이미지는 RGB이므로 3개의 채널과 32x32 픽셀입니다. 기본적으로 각 이미지는 0에서 255까지의 3 x 32 x 32 = 3072개의 숫자로 표현됩니다. <br>
신경망에서 일반적인 관행은 입력을 정규화하는 것인데, 일반적으로 사용되는 활성화 함수에서 포화를 피하고 수치적 안정성을 높이는 것을 포함한 여러 가지 이유로 수행됩니다. <br>
정규화 프로세스는 각 채널의 평균을 빼고 표준 편차로 나누는 것으로 구성됩니다. 텐서 "mean=[0.485, 0.456, 0.406]"과 "std=[0.229, 0.224, 0.225]"는 이미 계산되었으며, <br>
이는 훈련 세트로 의도된 CIFAR-10의 사전 정의된 하위 세트에서 각 채널의 평균과 표준 편차를 나타냅니다. 평균과 표준 편차를 처음부터 다시 계산하지 않고 테스트 세트에도 이러한 값을 사용하는 방법에 주목하십시오. 이는 네트워크가 위의 숫자를 뺀 후 나누어 생성된 특징을 기반으로 학습되었기 때문이며, 일관성을 유지하고자 하기 때문입니다. 또한, 실제로는 테스트 세트의 평균과 표준 편차를 계산할 수 없습니다. 왜냐하면 우리의 가정에 따르면 해당 시점에는 테스트 세트에 접근할 수 없기 때문입니다.

  ![CIFAR-10](https://github.com/ultralytics/docs/releases/download/0/cifar10-sample-image.avif)

  Data References:
    1. https://www.cs.toronto.edu/~kriz/cifar.html <br>
    2. https://developer-together.tistory.com/49 <br>
    3. https://tutorials.pytorch.kr/beginner/blitz/cifar10_tutorial.html?highlight=cifar

Training & testing data loading

In [30]:
import torch.utils


# batch size
batch_size = 128

# dataset for training
transform_train = transforms.Compose([
  transforms.ToTensor(),
  transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
])

train_dataset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_train)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)

# dataset for validation
transform_test = transforms.Compose([
  transforms.ToTensor(),
  transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
])

test_dataset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_test)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)


#### Model class 

TeaterNet : Deeper CNN

StudentNet : Lightweight CNN

In [31]:
# Deeper CNN class to be used as TeacherNet
class TeacherNet(nn.Module):
    def __init__(self, num_classes=10, dropout_rate=0.1):
        super().__init__()

        # feature extraction step : input (batch_size, 3, 32, 32) => output(batch_size, 32, 8, 8)
        self.features = nn.Sequential(
            nn.Conv2d(3, 128, kernel_size=3, padding=1),  
            nn.ReLU(),
            nn.Conv2d(128, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),            
        )  

        # flatten step : (batch_size, 32, 8, 8) → .view(batch_size, 2048) 또는 .flatten(1)

        # classifier step : (batch_size, 2048) → (batch_size, num_classes)
        self.classifier = nn.Sequential(
           nn.Linear(2048, 512),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1) # = x.view(x.size(0), -1)
        x = self.classifier(x)
        return x
   



In [32]:
# Lightweight neural network class to be used as student:
class StudentNet(nn.Module):
    def __init__(self, num_classes=10, dropout_rate=0.1):
        super().__init__()

        # feature extraction step : input (batch_size, 3, 32, 32) => output(batch_size, 16, 8, 8)
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(16, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
              
        # flatten step : (batch_size, 16, 8, 8) → .view(batch_size, 1024) 또는 .flatten(1)

        self.classifier = nn.Sequential(
            nn.Linear(1024, 256),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x

##### TeacherNet, StudentNet Parameters

각 모델의 파라미터 수를 출력하여 비교한다.

In [33]:
def count_parameters(model):
  return sum(p.numel() for p in model.parameters())

teacher_net = TeacherNet(num_classes=100)
student_net = StudentNet(num_classes=100)

print(f"TeacherNet has {count_parameters(teacher_net):,} trainable parameters")
print(f"StudentNet has {count_parameters(student_net):,} trainable parameters")

TeacherNet has 1,233,156 trainable parameters
StudentNet has 290,868 trainable parameters


#### TeacherNet vs StudentNet

TeacherNet 과 StudentNet 을 각각 학습하여 성능을 비교한다.

In [34]:
def train(model, train_loader, epochs, learning_rate, device):
  # loss function and optimizer
  criterion = nn.CrossEntropyLoss()
  optimizer = optim.Adam(model.parameters(), lr=learning_rate)

  model.train()

  for epoch in range(epochs):
    running_loss = 0.0

    # inputs: A collection of batch_size images
    # labels: A vector of dimensionality batch_size with integers denoting class of each image
    for inputs, labels in train_loader:
      # Move data to device
      inputs, labels = inputs.to(device), labels.to(device)

      optimizer.zero_grad()

      # forward pass
      outputs = model(inputs)

      # outputs: Output of the network for the collection of images. A tensor of dimensionality batch_size x num_classes
      # labels: The actual labels of the images. Vector of dimensionality batch_size
      # criterion: The Cross-Entropy loss function
      loss = criterion(outputs, labels)
      loss.backward()

      # update weights
      optimizer.step()

      running_loss += loss.item()

    print(f"Epoch {epoch+1}/{epochs}, Loss: {running_loss/len(train_loader)}")


def test(model, test_loader, device):
  model.eval()

  correct = 0
  total = 0

  with torch.no_grad():
    for inputs, labels in test_loader:
      # Move data to device
      inputs, labels = inputs.to(device), labels.to(device)

      # forward pass
      outputs = model(inputs)

      _, predicted = torch.max(outputs.data, 1)
      
      total += labels.size(0)
      correct += (predicted == labels).sum().item()

  accuracy = 100 * correct / total
  print(f"Accuracy of the model on the test set: {accuracy:.2f}%")

  return accuracy


#### TeacherNet training

In [35]:
torch.manual_seed(42)

teacher_net = TeacherNet(num_classes=10).to(my_device)
train(teacher_net, train_loader, epochs=10, learning_rate=0.001, device=my_device)

# test accuracy
teacher_test_accuracy = test(teacher_net, test_loader, device=my_device)

Epoch 1/10, Loss: 1.3282756302362817
Epoch 2/10, Loss: 0.8602076454845535
Epoch 3/10, Loss: 0.6802018845020352
Epoch 4/10, Loss: 0.5306509268253355
Epoch 5/10, Loss: 0.4088569336839954
Epoch 6/10, Loss: 0.3117652456931141
Epoch 7/10, Loss: 0.22789609557984736
Epoch 8/10, Loss: 0.1732932399110416
Epoch 9/10, Loss: 0.13821860919694615
Epoch 10/10, Loss: 0.11614861560370916
Accuracy of the model on the test set: 75.34%


In [36]:
#### StudentNet instantiation
torch.manual_seed(42)
student_net = StudentNet(num_classes=10).to(my_device)

torch.manual_seed(42)
new_student_net = StudentNet(num_classes=10).to(my_device)

# 두 네트웍이 같은지 첫번째 레이어의 가중치 norm 을 출력
print("Norm of 1st layer of student_net:", torch.norm(student_net.features[0].weight).item())
print("Norm of 1st layer of new student_net:", torch.norm(new_student_net.features[0].weight).item())

Norm of 1st layer of student_net: 2.327361822128296
Norm of 1st layer of new student_net: 2.327361822128296


#### StudentNet training

In [37]:
train(student_net, train_loader, epochs=10, learning_rate=0.001, device=my_device)
student_test_accuracy = test(student_net, test_loader, device=my_device)

Epoch 1/10, Loss: 1.4706902052740307
Epoch 2/10, Loss: 1.1598624497118508
Epoch 3/10, Loss: 1.0282264758863717
Epoch 4/10, Loss: 0.9261842036186276
Epoch 5/10, Loss: 0.8499267397024443
Epoch 6/10, Loss: 0.7822678686712708
Epoch 7/10, Loss: 0.7169250473189537
Epoch 8/10, Loss: 0.6588636029254445
Epoch 9/10, Loss: 0.6045040360192204
Epoch 10/10, Loss: 0.555988242863999
Accuracy of the model on the test set: 70.17%


##### Test Accuracy

In [38]:
print(f"TeacherNet accuracy: {teacher_test_accuracy:.2f}%")
print(f"StudentNet accuracy: {student_test_accuracy:.2f}%")

TeacherNet accuracy: 75.34%
StudentNet accuracy: 70.17%


## Knowledge Distillation (지식 증류)

#### 1st Method : 

<font color="green">**교사 모델의 출력 분포(soft targets)를 학습**</font>

지식 증류(knowledge distillation)는 두 네트워크 모두 클래스에 대해 확률 분포를 출력한다는 사실에 기반한 간단한 기법입니다. 

기존 Cross Entropy 손실에 교사 네트워크의 소프트맥스 출력을 기반으로 추가적인 손실로 통합함으로서 구현합니다. 

이는 적절하게 훈련된 교사 네트워크의 출력 활성화가 학습 중에 학생 네트워크가 활용할 수 있는 추가 정보를 전달한다는 가정에 기반합니다.

지식 증류에서는 **soft target** 의 전체 분포, 특히 작은 확률로 나타난 비정답 클래스 정보를 활용하면 모델이 학습 데이터의 내재된 구조와 클래스 간 관계를 더 잘 이해할 수 있고, 결과적으로 더 좋은 일반화 성능을 가지게 됩니다.


![1st Method](https://docs.pytorch.org/tutorials/_static/img/knowledge_distillation/distillation_output_loss.png)

----

## Soft Targets Loss

#### (1) soft target 이란 ?

> **soft target** : 기존의 잘 학습된 큰 모델 (teacher)의 출력 <br>
<font color=skyblue>soft target = [고양이: 0.01, 개: 0.02, 자동차: 0.10, 트럭: 0.75, 비행기: 0.08, ...]</font> <br>
**hard target** : 보통 정답 하나만 있는 실제 label. 예: label = 3 (트럭)


##### (2) soft target 에서 작은 확률 값이 중요한가 ?

> 트럭(정답): 0.75, 자동차: 0.10, 비행기: 0.08, 고양이: 0.01 <br>
<font color=orange>**이 작은 값들이 알려주는 것은 트럭과 다른 클래스들 간의 유사성 정보입니다.**</font> <br>
자동차, 비행기 → 트럭과 시각적 유사성 있음 (교통수단, 바퀴 등) → 상대적으로 확률이 높음 <br>
고양이, 개 → 완전히 다른 클래스 → 확률이 거의 없음


"더 작은 확률" 값은 정답 클래스가 아니더라도, 그 확률이 얼마나 나왔는지를 통해 모델이 다른 클래스들과 얼마나 혼동하고 있는지를 알 수 있다는 것입니다. <br>
예로 모델이 트럭과 자동차를 혼동했다면 → 둘이 비슷한 특성이 있다는 뜻입니다. 이런 정보가 학습 데이터의 구조적 유사성을 잘 보존하는 데 기여합니다.


##### (3) soft target을 이용하면 왜 도움이 되나 ?

단순히 정답(hard target)만 학습하면 정답만 맞추고, 다른 클래스들과의 관계는 무시됩니다. 하지만 soft target 까지 학습하면 <br>

> <font color=orange>**"트럭 이라는 정답뿐만 아니라 자동차와 비행기도 좀 비슷하니까 헷갈릴 수 있어"**</font> 라는 클래스 간의 구조적 유사성을 모델이 배움으로서 더 일반화된 모델이 될 수 있습니다.


##### (4) Teacher vs Student 사이의 확률 분포 차이를 손실로 ?

KL Divergence를 이용하여 Teacher, Student 모델의 출력 값인 확률 분포의 차이 (soft targets loss)를 계산합니다. <br>
> <font color=green>**KL Divergence (Kullback–Leibler Divergence)는 확률 분포 간의 차이를 측정할 때 사용됩니다.**</font> <br>
> <br>
> KL(P∥Q)= ∑ P(i)log(P(i)/Q(i)) = ∑ P(i)(log(P(i)) - log(Q(i))) <br>
> <br>
𝑃 : 참(true) 확률 분포 (target) - Teacher Net의 결과 확률 분포 <br>
Q : 근사한 확률 분포 (prediction) - Student Net의 결과 확률 분포

➡ 이 식은 Q가 P를 얼마나 잘 설명하지 못하는지를 측정합니다. 즉, Q가 P와 다를수록 KL 값이 커집니다.

<br>

<font color=orange>**결과적으로 이 KL Divergence 를 기반으로 Loss로 정의하여 학습하면 Student는 Teacher를 잘 학습하게 됩니다.**</font>





In [39]:
""" 
  T : temperature : T 는 출력 분포의 부드러움을 제어합니다. 

  신경망의 마지막 출력(logits) 는 softmax 를 통해 클래스 확률로 바뀝니다.  
  
  softmax 는 가장 큰 값이 있는 위치에 거의 모든 확률을 몰아주는 경향이 있기 때문에 결과가 hard target 에 가까워집니다.
  
  지식 증류에서는 softmax 계산 시, 로짓을 온도 T로 나눠서 "더 부드러운" 분포를 만듭니다.

  T = 1	: 정답에 집중됨
  𝑇 > 1 : 분포가 평평해짐(부드러워짐)
  T < 1 : 분포가 더 날카롭게 변함 (확신 강함)
"""

#  teacher net >>> student net 으로 지식 증류
def train_knowledge_distillation(teacher_net, student_net, train_loader, epochs, learning_rate, 
                                 T, soft_loss_weight, hard_loss_weight, device):

  # loss function and optimizer for target student network
  criterion = nn.CrossEntropyLoss()
  optimizer = optim.Adam(student_net.parameters(), lr=learning_rate)

  # teacher set to evaluation mode
  teacher_net.eval()

  # student set to training mode
  student_net.train()

  for epoch in range(epochs):
    running_loss = 0.0

    for inputs, labels in train_loader:
      # Move data to device
      inputs, labels = inputs.to(device), labels.to(device)

      optimizer.zero_grad()

      # Forward pass with the teacher model - do not save gradients here as we do not change the teacher's weights
      with torch.no_grad():
        teacher_logits = teacher_net(inputs)      

      # forward pass
      student_logits = student_net(inputs)

      # total loss = weighted CrossEntropy(hard loss) + weighted KL_Divergence(soft loss)
      # Soften the teacher logits by applying softmax first
      teacher_soft_logits = nn.functional.softmax(teacher_logits / T, dim=1)

      # Soften the student logits by applying softmax first and log() second
      student_soft_logits = nn.functional.log_softmax(student_logits / T, dim=1)


      # Calculate the soft targets loss with KL divergence. 
      # Distillation 논문(Hinton et al., 2015)에서 온도 𝑇를 도입하면 loss 값이 작아져 Gradient Scale 도 작아짐
      # 따라서 최종 softloss 에서는  T**를 곱해 줌으로써 스케일을 복원합니다
      # soft_targets_loss = torch.sum(teacher_soft_logits * (teacher_soft_logits.log() - student_soft_logits)) / student_soft_logits.size()[0] * (T**2)
      soft_targets_loss = nn.functional.kl_div(student_soft_logits, teacher_soft_logits, reduction='batchmean') * (T ** 2)

      # Calculate the label loss (hard loss)
      hard_targets_loss = criterion(student_logits, labels)

      # Weighted sum of the two losses : CE + KD
      loss = soft_loss_weight * soft_targets_loss + hard_loss_weight * hard_targets_loss


      loss.backward()
      optimizer.step()

      running_loss += loss.item()

    print(f"Epoch {epoch+1}/{epochs}, Loss: {running_loss/len(train_loader)}")


In [40]:
# Apply ``train_knowledge_distillation`` with a temperature of 2. Arbitrarily set the weights to 0.75 for CE and 0.25 for distillation loss.
train_knowledge_distillation(teacher_net=teacher_net, student_net=new_student_net, train_loader=train_loader, epochs=10, learning_rate=0.001, 
                             T=2, soft_loss_weight=0.25, hard_loss_weight=0.75, device=my_device)

kd_student_test_accuracy = test(new_student_net, test_loader, my_device)

# Compare the student test accuracy with and without the teacher, after distillation
print(f"Teacher accuracy: {teacher_test_accuracy:.2f}%")
print(f"Student accuracy without teacher: {student_test_accuracy:.2f}%")
print(f"Student accuracy with CE + KD: {kd_student_test_accuracy:.2f}%")

Epoch 1/10, Loss: 2.4000869175357282
Epoch 2/10, Loss: 1.8840148220281772
Epoch 3/10, Loss: 1.661236142868276
Epoch 4/10, Loss: 1.5036588898095329
Epoch 5/10, Loss: 1.3754271006645145
Epoch 6/10, Loss: 1.2605154997552448
Epoch 7/10, Loss: 1.1669648294253727
Epoch 8/10, Loss: 1.0771046384521152
Epoch 9/10, Loss: 1.005654371912827
Epoch 10/10, Loss: 0.9352872999732756
Accuracy of the model on the test set: 70.76%
Teacher accuracy: 75.34%
Student accuracy without teacher: 70.17%
Student accuracy with CE + KD: 70.76%


### 2nd Method

<font color="green">**교사 모델 내부의 Hidden Representation을 학습**</font>

지금부터는 출력 계층보다는 숨겨진 상태에 집중해 보겠습니다. <br>
이 방법의 근거는 교사 모델이 외부 개입 없이는 학생 모델이 달성하기 어려울 정도로 더 나은 내부 표현을 가지고 있다는 가정 하에 진행됩니다. <br>
따라서 학생 모델이 교사 모델의 내부 표현을 모방하도록 인위적으로 강제하지만 이것이 학생에게 도움이 될지는 명확하지 않습니다. <br>
왜냐하면 네트워크의 아키텍처가 다르고 학생 모델의 학습 능력이 교사 모델과 동일하지 않기 때문에 해로울 수도 있기 때문입니다.

이 방법의 영향을 파악하기 위해 CosineEmbeddingLoss 함수를 이용하여 간단한 실험을 수행할 수 있습니다.

----
<br>

##### (1) CosineEmbeddingLoss

**두 벡터의 방향(코사인 유사도) 을 비교하는 손실 함수**입니다.

**목적**: 두 벡터가 비슷한 방향을 가지도록 학습 유도 <br>
**적용**: 두 표현 벡터 (teacher 와 student 의 hidden representation) 을 비교할 때 자주 사용

##### (2) CosineEmbeddingLoss 수식

> ***loss(x1, x2, y) = if y =  1,  1−cos(x1, x2),   if y = -1,  max(0, cos(x1, x2)-margin)***
>
> (a) x1, x2: 비교 대상 벡터들 (예: student와 teacher의 feature vector) <br>
(b) y : 라벨 (1: 유사하길 원함, -1: 다르길 원함) <br>
(c) cos(x1, x2) : 두 벡터의 코사인 유사도 <br>
(d) margin : y = -1 일 때, 최소한 이 정도는 다르기를 바라는 기준


Knowledge distillation에서는 일반적으로 y = 1로 사용하여 student와 teacher의 representation이 유사하길 원합니다.


##### (3) CosineEmbeddingLoss vs CrossEntropy

<font color="orange">CrossEntopy 는 정답 클래스에만 집중하지만 CosineEmbeddingLoss 는 전체 표현 공간 상에서의 유사성을 학습 시킵니다.</font>

특히 서로 다른 구조의 모델 간 표현을 정렬(alignment: 두 벡터가 같은 방향일 때, 유사도가 큼) 싶을 때, 유용합니다.
















----

Teacher Model Modification

차원이 다른 두 모델의 Hidden Representation 을 일치시키기 위해 Teacher Model 에 Pooling 을 새롭게 추가합니다.

In [41]:
# Deeper CNN class to be used as TeacherNet
class Modified_TeacherNet(nn.Module):
  def __init__(self, num_classes=10, dropout_rate=0.1):
    super().__init__()

    # feature extraction step : input (batch_size, 3, 32, 32) => output(batch_size, 32, 8, 8)
    self.features = nn.Sequential(
      nn.Conv2d(3, 128, kernel_size=3, padding=1),  
      nn.ReLU(),
      nn.Conv2d(128, 64, kernel_size=3, padding=1),
      nn.ReLU(),
      nn.MaxPool2d(kernel_size=2, stride=2),
      nn.Conv2d(64, 64, kernel_size=3, padding=1),
      nn.ReLU(),
      nn.Conv2d(64, 32, kernel_size=3, padding=1),
      nn.ReLU(),
      nn.MaxPool2d(kernel_size=2, stride=2),            
    )  

    # flatten step : (batch_size, 32, 8, 8) → .view(batch_size, 2048) 또는 .flatten(1)

    # classifier step : (batch_size, 2048) → (batch_size, num_classes)
    self.classifier = nn.Sequential(
      nn.Linear(2048, 512),
      nn.ReLU(),
      nn.Dropout(dropout_rate),
      nn.Linear(512, num_classes)
    )

  def forward(self, x):
    x = self.features(x)
    flattened_conv_output = torch.flatten(x, 1) # = x.view(x.size(0), -1)
    x = self.classifier(flattened_conv_output)

    # student model 의 hidden representation 차원 : (batch_size, 1024)
    # teacher model 의 hidden representation 차원 : (batch_size, 2048)
    # 두 모델의 hidden representation 차원이 다르기 때문에 두 모델의 hidden representation 을 일치시키기 위해 
    # teacher model 의 hidden representation 을 평균 풀링(average pooling) 을 통해 차원을 줄입니다.
    flattened_conv_output_after_pooling = torch.nn.functional.avg_pool1d(flattened_conv_output, 2)

    return x, flattened_conv_output_after_pooling


In [42]:
# Create a similar student class where we return a tuple. We do not apply pooling after flattening.
class Modified_StudentNet(nn.Module):
  def __init__(self, num_classes=10, dropout_rate=0.1):
    super().__init__()

    # feature extraction step : input (batch_size, 3, 32, 32) => output(batch_size, 16, 8, 8)
    self.features = nn.Sequential(
      nn.Conv2d(3, 16, kernel_size=3, padding=1),
      nn.ReLU(),
      nn.MaxPool2d(kernel_size=2, stride=2),
      nn.Conv2d(16, 16, kernel_size=3, padding=1),
      nn.ReLU(),
      nn.MaxPool2d(kernel_size=2, stride=2),
    )
          
    # flatten step : (batch_size, 16, 8, 8) → .view(batch_size, 1024) 또는 .flatten(1)

    self.classifier = nn.Sequential(
      nn.Linear(1024, 256),
      nn.ReLU(),
      nn.Dropout(dropout_rate),
      nn.Linear(256, num_classes)
    )

  def forward(self, x):
    x = self.features(x)
    flattened_conv_output = torch.flatten(x, 1)
    x = self.classifier(flattened_conv_output)
    return x, flattened_conv_output

In [43]:
# We do not have to train the modified deep network from scratch of course, we just load its weights from the trained instance
modified_teacher_net = Modified_TeacherNet(num_classes=10).to(my_device)
modified_teacher_net.load_state_dict(teacher_net.state_dict())

# Once again ensure the norm of the first layer is the same for both networks
print("Norm of 1st layer for teacher net:", torch.norm(teacher_net.features[0].weight).item())
print("Norm of 1st layer for modified teacher net:", torch.norm(modified_teacher_net.features[0].weight).item())

# Initialize a modified lightweight network with the same seed as our other lightweight instances. This will be trained from scratch to examine the effectiveness of cosine loss minimization.
torch.manual_seed(42)
modified_student_net = Modified_StudentNet(num_classes=10).to(my_device)
print("Norm of 1st layer:", torch.norm(modified_student_net.features[0].weight).item())

Norm of 1st layer for teacher net: 7.505339622497559
Norm of 1st layer for modified teacher net: 7.505339622497559
Norm of 1st layer: 2.327361822128296


Check model output(the logits, hidden_representation)

In [44]:
# Create a sample input tensor
sample_input = torch.randn(128, 3, 32, 32).to(my_device) # (batch_size, channels, height, width)

# Pass the input through the modified teacher model
logits, hidden_representation = modified_teacher_net(sample_input)

print(f"Teacher logits shape: {logits.shape}")
print(f"Teacher hidden_representation shape: {hidden_representation.shape}")

# Pass the input through the modified student model
modified_logits, modified_hidden_representation = modified_student_net(sample_input)

print(f"Student logits shape: {modified_logits.shape}")
print(f"Student hidden_representation shape: {modified_hidden_representation.shape}")







Teacher logits shape: torch.Size([128, 10])
Teacher hidden_representation shape: torch.Size([128, 1024])
Student logits shape: torch.Size([128, 10])
Student hidden_representation shape: torch.Size([128, 1024])


#### Modified Training Pass

Teacher Models' Hidden Representation 2048 차원이 1024 로 축소되며 발생하는 손실은 손실 계산전 Student Model 가중치에만 영향을 미칩니다.

즉 Student Model's Classifier 에는 영향을 미치지 않습니다. 이에 맞게 학습 루틴도 다음과 같이 새롭게 구현합니다.

![2nd method](https://docs.pytorch.org/tutorials/_static/img/knowledge_distillation/cosine_loss_distillation.png)

##### Train function using the cosine embedding loss

In [45]:
def train_cosine_loss(teacher_net, student_net, train_loader, epochs, learning_rate, 
                      hidden_representation_loss_weight, cross_entropy_loss_weight, device):

  cross_entropy_loss = nn.CrossEntropyLoss()
  cosine_embedding_loss = nn.CosineEmbeddingLoss()
  optimizer = optim.Adam(student_net.parameters(), lr=learning_rate)
  
  student_net.train()

  for epoch in range(epochs):
    running_loss = 0.0

    for inputs, labels in train_loader:
      # Move data to device
      inputs, labels = inputs.to(device), labels.to(device)

      # Zero the parameter gradients
      optimizer.zero_grad()

      # Forward pass with the teacher model - do not save gradients here as we do not change the teacher's weights
      with torch.no_grad():
        _, teacher_hidden_representation = teacher_net(inputs)

      # forward pass
      student_logits, student_hidden_representation = student_net(inputs)

      # Calculate the cosine embedding loss. Target is a vector of ones. 
      # From the loss formula above we can see that is the case where loss minimization leads to cosine similarity increase.
      hidden_representation_loss = cosine_embedding_loss(student_hidden_representation, teacher_hidden_representation, 
                                                         target=torch.ones(inputs.size(0)).to(device))
      
      # Calculate the cross entropy loss : true label loss
      label_loss = cross_entropy_loss(student_logits, labels)

      # Calculate the total loss : weighted sum of hidden representation loss and cross entropy loss
      loss = hidden_representation_loss_weight * hidden_representation_loss + cross_entropy_loss_weight * label_loss

      # Backward pass and update weights
      loss.backward()
      optimizer.step()

      running_loss += loss.item()

    print(f"Epoch {epoch+1}/{epochs}, Loss: {running_loss/len(train_loader)}")



In [46]:
def test_multiple_outputs(model, test_loader, device):
  model.eval()

  correct = 0
  total = 0

  with torch.no_grad():
    for inputs, labels in test_loader:
      # Move data to device
      inputs, labels = inputs.to(device), labels.to(device)

      # Forward pass
      outputs, _ = model(inputs)

      # Get the predicted class with the highest probability
      _, predicted = torch.max(outputs.data, 1)

      total += labels.size(0)

      # Count the number of correct predictions
      correct += (predicted == labels).sum().item()

  accuracy = 100 * correct / total
  print(f"Accuracy of the model on the test set: {accuracy:.2f}%")

  return accuracy



지식 증류를 위한 1st method 와 2nd method 를 쉽게 결합할 수 있습니다. 일반적으로 Teacher, Student 패러다임에서 더 나은 성능을 얻기 위해

여러 방법들을 결합합니다. 하지만 지금은 간단하게 학습-테스트를 실행해 보겠습니다.

In [47]:
# Train and test the lightweight network with cross entropy loss
train_cosine_loss(teacher_net=modified_teacher_net, student_net=modified_student_net, train_loader=train_loader, epochs=10, learning_rate=1e-3, 
                  hidden_representation_loss_weight=0.25, cross_entropy_loss_weight=0.75, device=my_device)

modified_student_test_accuracy = test_multiple_outputs(modified_student_net, test_loader, device=my_device)
print(f"Modified student test accuracy: {modified_student_test_accuracy:.2f}%")


Epoch 1/10, Loss: 1.3009181461675698
Epoch 2/10, Loss: 1.0663385705264938
Epoch 3/10, Loss: 0.9659812936697469
Epoch 4/10, Loss: 0.8900845317584475
Epoch 5/10, Loss: 0.836711060513011
Epoch 6/10, Loss: 0.7911873215909504
Epoch 7/10, Loss: 0.7537756514976092
Epoch 8/10, Loss: 0.7156240833384911
Epoch 9/10, Loss: 0.6800502843564123
Epoch 10/10, Loss: 0.6549125130828994
Accuracy of the model on the test set: 70.91%
Modified student test accuracy: 70.91%


### 3rd Method

<font color=green>회귀 분석(regression analysis)를 이용한 **교사 모델 내부의 Hidden Representation 학습**</font>

이전(2nd method)의 단순한 최소화 방법으로는 여러 가지 이유로 더 나은 결과를 보장하지 못하는데, 그 중 하나는 벡터의 차원입니다. 일반적으로 고차원 벡터의 경우 코사인 유사도가 유클리드 거리보다 더 잘 작동하지만, 각각 1024개의 구성 요소를 가진 벡터를 다루었기 때문에 의미 있는 유사도를 추출하기가 훨씬 더 어렵습니다. 또한, 앞서 언급했듯이 <. color=red>교사와 학생의 은닉 표현을 일치시키는 것은 이론적으로 뒷받침되지 않습니다</font>. 이러한 벡터의 1:1 일치를 목표로 삼아야 할 타당한 이유는 없습니다. 

이를 개선하기 위해 회귀기(regressor)라는 추가 네트워크를 이용하여 지식 증류 학습 방법을 개선하려고 합니다. 목표는 2nd 방법 처럼 먼저 합성곱 계층 이후에 교사 모델의 특징 맵을 추출하고, 다음으로 합성곱 계층 이후에 학생 모델의 특징 맵을 추출한 후, 마지막으로 이 맵들을 일치시키는 것입니다. 그러나 이번에는 네트워크 사이에 ***회귀기***를 도입하여 일치 과정을 용이하게 할 것입니다. ***회귀기는 학습 가능하며 이상적으로는 우리의 단순 코사인 손실 최소화 방식보다 더 나은 성능을 보일 것입니다***. 

이 Regressor 모델의 주요 역할은 교사와 학생 모델 간의 손실 함수를 적절하게 정의할 수 있도록 특징 맵 차원을 일치시키는 것입니다. <font color=orange>이러한 손실 함수를 정의하면 학습 "경로"가 생성되는데, 이는 기본적으로 학생의 가중치를 변경하는 그래디언트를 역전파하는 흐름입니다</font>. 기존 네트워크의 각 분류기 바로 앞에 있는 합성곱 계층의 출력에 초점을 맞추면 다음과 같은 모양이 됩니다.


In [48]:
# Pass the sample input only from the convolutional feature extractor
convolutional_fe_output_student = modified_student_net.features(sample_input)
convolutional_fe_output_teacher = modified_teacher_net.features(sample_input)

# Print their shapes
print("Student's feature extractor output shape: ", convolutional_fe_output_student.shape)
print("Teacher's feature extractor output shape: ", convolutional_fe_output_teacher.shape)

Student's feature extractor output shape:  torch.Size([128, 16, 8, 8])
Teacher's feature extractor output shape:  torch.Size([128, 32, 8, 8])


교사용 필터는 32개, 학생용 필터는 16개입니다. ***학생의 특징 맵을 교사의 특징 맵 형태로 변환하는 학습 가능한 계층을 추가합니다.***. 실제로는 합성곱 특징 맵의 크기에 맞는 중간 회귀 분석기를 거친 후 은닉 상태를 반환하도록 경량 클래스를 수정하고, 교사 클래스는 풀링이나 평탄화 없이 최종 합성곱 계층의 출력을 반환하도록 수정합니다.

![3rd method](https://docs.pytorch.org/tutorials/_static/img/knowledge_distillation/fitnets_knowledge_distill.png)

In [49]:
# Deeper CNN class to be used as TeacherNet
class Modified_TeacherNetForRegressor(nn.Module):
  def __init__(self, num_classes=10, dropout_rate=0.1):
    super().__init__()

    # feature extraction step : input (batch_size, 3, 32, 32) => output(batch_size, 32, 8, 8)
    self.features = nn.Sequential(
      nn.Conv2d(3, 128, kernel_size=3, padding=1),  
      nn.ReLU(),
      nn.Conv2d(128, 64, kernel_size=3, padding=1),
      nn.ReLU(),
      nn.MaxPool2d(kernel_size=2, stride=2),
      nn.Conv2d(64, 64, kernel_size=3, padding=1),
      nn.ReLU(),
      nn.Conv2d(64, 32, kernel_size=3, padding=1),
      nn.ReLU(),
      nn.MaxPool2d(kernel_size=2, stride=2),            
    )  

    # flatten step : (batch_size, 32, 8, 8) → .view(batch_size, 2048) 또는 .flatten(1)

    # classifier step : (batch_size, 2048) → (batch_size, num_classes)
    self.classifier = nn.Sequential(
      nn.Linear(2048, 512),
      nn.ReLU(),
      nn.Dropout(dropout_rate),
      nn.Linear(512, num_classes)
    )

  def forward(self, x):
    x = self.features(x)
    conv_feature_map = x

    x = torch.flatten(x, 1) # = x.view(x.size(0), -1)
    x = self.classifier(x)

    return x, conv_feature_map




class Modified_StudentNetForRegressor(nn.Module):
  def __init__(self, num_classes=10, dropout_rate=0.1):
    super().__init__()

    # feature extraction step : input (batch_size, 3, 32, 32) => output(batch_size, 16, 8, 8)
    self.features = nn.Sequential(
      nn.Conv2d(3, 16, kernel_size=3, padding=1),
      nn.ReLU(),
      nn.MaxPool2d(kernel_size=2, stride=2),
      nn.Conv2d(16, 16, kernel_size=3, padding=1),
      nn.ReLU(),
      nn.MaxPool2d(kernel_size=2, stride=2),
    )

    # Include an extra regressor (in our case linear)
    # (batch_size, 16, 8, 8) => (batch_size, 32, 8, 8)
    self.regressor = nn.Sequential(
      nn.Conv2d(16, 32, kernel_size=3, padding=1)
    )      

    # flatten step : (batch_size, 16, 8, 8) → .view(batch_size, 1024) 또는 .flatten(1)

    self.classifier = nn.Sequential(
      nn.Linear(1024, 256),
      nn.ReLU(),
      nn.Dropout(dropout_rate),
      nn.Linear(256, num_classes)
    )

  def forward(self, x):
    x = self.features(x)
    regressor_output = self.regressor(x)
    
    x = torch.flatten(x, 1)
    x = self.classifier(x)

    return x, regressor_output
    

##### Train function using the MSE loss

In [50]:
def train_mse_loss(teacher_net, student_net, train_loader, epochs, learning_rate, 
                   feature_map_loss_weight, cross_entropy_loss_weight, device):
  
  mse_loss = nn.MSELoss()
  ce_loss = nn.CrossEntropyLoss()
  optimizer = torch.optim.Adam(student_net.parameters(), lr=learning_rate)

  teacher_net.to(device)
  student_net.to(device)

  teacher_net.eval()
  student_net.train()

  for epoch in range(epochs):
    running_loss = 0.0

    for images, labels in train_loader:
      # Move data to device
      images, labels = images.to(device), labels.to(device)

      # Zero the parameter gradients
      optimizer.zero_grad()

      # Forward pass for teacher
      with torch.no_grad():
        _, teacher_feature_map = teacher_net(images)

      # Forward pass for student
      student_output, student_regressor_output = student_net(images)

      # Calculate the loss for Knowledge Distillation
      feature_map_loss = mse_loss(student_regressor_output, teacher_feature_map)

      # Calculate the loss for Cross Entropy
      label_loss = ce_loss(student_output, labels)

      # Weighted sum of the losses
      loss = feature_map_loss_weight * feature_map_loss + cross_entropy_loss_weight * label_loss
      
      # Backward pass and optimization
      loss.backward()
      optimizer.step()

      running_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {running_loss / len(train_loader)}")


In [51]:
# Notice how our test function remains the same here with the one we used in our previous case. We only care about the actual outputs because we measure accuracy.

# Initialize a Modified_StudentNetForRegressor
torch.manual_seed(42)
modified_reg_student_net = Modified_StudentNetForRegressor(num_classes=10).to(my_device)


# We do not have to train the modified deep network from scratch of course, we just load its weights from the trained instance
modified_reg_teacher_net = Modified_TeacherNetForRegressor(num_classes=10).to(my_device)
modified_reg_teacher_net.load_state_dict(teacher_net.state_dict())

# Train and test once again
train_mse_loss(teacher_net=modified_reg_teacher_net, student_net=modified_reg_student_net, train_loader=train_loader, epochs=10, learning_rate=0.001, 
               feature_map_loss_weight=0.25, cross_entropy_loss_weight=0.75, device=my_device)

modified_reg_student_test_accuracy = test_multiple_outputs(modified_reg_student_net, test_loader, my_device)

Epoch 1, Loss: 1.7269266548059177
Epoch 2, Loss: 1.3524827316898824
Epoch 3, Loss: 1.2069865216684463
Epoch 4, Loss: 1.1097243466340672
Epoch 5, Loss: 1.0316116919602885
Epoch 6, Loss: 0.9663170795611409
Epoch 7, Loss: 0.9109575531976607
Epoch 8, Loss: 0.8600117003216463
Epoch 9, Loss: 0.8180607978035422
Epoch 10, Loss: 0.7784848411369811
Accuracy of the model on the test set: 71.98%


In [52]:
print(f"Teacher accuracy: {teacher_test_accuracy:.2f}%")
print(f"Student accuracy without teacher: {student_test_accuracy:.2f}%")
print(f"Student accuracy with CE + KD: {kd_student_test_accuracy:.2f}%")
print(f"Student accuracy with CE + CosineLoss: {modified_student_test_accuracy:.2f}%")
print(f"Student accuracy with CE + RegressorMSE: {modified_reg_student_test_accuracy:.2f}%")

Teacher accuracy: 75.34%
Student accuracy without teacher: 70.17%
Student accuracy with CE + KD: 70.76%
Student accuracy with CE + CosineLoss: 70.91%
Student accuracy with CE + RegressorMSE: 71.98%
